# Анализ эксперимента по подстройке коэффициентов Kp и Ki от 25 ноября 2025


Используется версия 4.0.0, в который были заменены все TorchRL компоненты.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nn_laser_stabilizer.config.config import load_config
from nn_laser_stabilizer.utils.paths import get_experiment_dir

EXPERIMENT_NAME = "pid_delta_tuning"
EXPERIMENT_DATE = "2025-11-25"
EXPERIMENT_TIME = "15-51-58"

EXPERIMENT_DIR = get_experiment_dir(
    experiment_name=EXPERIMENT_NAME,
    experiment_date=EXPERIMENT_DATE,
    experiment_time=EXPERIMENT_TIME,
)
CONFIG_PATH = EXPERIMENT_DIR / "config.yaml"
ENV_LOG_PATH = EXPERIMENT_DIR / "env_logs" / "env.log"
TRAIN_LOG_PATH = EXPERIMENT_DIR / "logs" / "train.log"
CONNECTION_LOG_PATH = EXPERIMENT_DIR / "connection_logs" / "connection.log"

config = load_config(CONFIG_PATH)
print(f"Эксперимент: {config.experiment_name}")
print(f"Seed: {config.seed}")

## Анализ логов окружения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_keyval_log

# env.log содержит строки двух типов: шаги (есть should_reset) и reset (нет).
# parse_keyval_log читает все строки; ниже воспроизводим прежнюю логику:
#   - классификация step/reset по наличию should_reset;
#   - reset-строки наследуют номер последнего шага (как current_step).
_raw = parse_keyval_log(ENV_LOG_PATH)
_raw = _raw[
    _raw[["kp", "ki", "kd", "error_mean_norm", "error_std_norm"]].notna().all(axis=1)
].reset_index(drop=True)

is_step = _raw["should_reset"].notna()

env_df = pd.DataFrame({
    "Step": _raw["step"],
    "time": _raw["time"],
    "Type": np.where(is_step, "step", "reset"),
    "Kp": _raw["kp"],
    "Ki": _raw["ki"],
    "Kd": _raw["kd"],
    "Delta Kp": _raw["delta_kp_norm"].where(is_step),
    "Delta Ki": _raw["delta_ki_norm"].where(is_step),
    "Error mean norm": _raw["error_mean_norm"],
    "Error std norm": _raw["error_std_norm"],
    "Reward": _raw["reward"].where(is_step),
    "Should reset": _raw["should_reset"].where(is_step, other=True).astype(bool),
})
env_df["Step"] = env_df["Step"].ffill().fillna(0).astype(int)
reset_steps = env_df.loc[env_df["Type"] == "reset", "Step"].tolist()

print(f"Загружено {len(env_df)} записей из логов окружения")
print(f"Найдено {len(reset_steps)} reset событий")
print(f"Диапазон шагов: {env_df['Step'].min()} - {env_df['Step'].max()}")

In [ ]:
step_df = env_df[env_df['Type'] == 'step'].copy()
print("=== Статистика по шагам окружения ===")
print(step_df.describe())


In [ ]:
exploration_steps = config.training.exploration_steps
initial_collect_steps = config.training.initial_collect_steps
neural_network_step = max(initial_collect_steps, exploration_steps) - 100 # этап warmup

columns_to_plot = ['Kp', 'Ki', 'Error mean norm', 'Error std norm', 'Reward']

for col in columns_to_plot:
    plt.figure(figsize=(12, 5))
    plt.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=0.8, label=col)
    
    for reset_step in reset_steps:
        if reset_step <= step_df['Step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=2, alpha=0.6)
    
    if neural_network_step <= step_df['Step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN')
    
    plt.title(f'{col} over Steps')
    plt.xlabel('Step')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

### График для встречи 26 декабря 2025 г

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Настройка
plt.rcParams.update({'font.size': 14})  # крупный шрифт для слайда

neural_network_step = max(initial_collect_steps, exploration_steps) - 100 # этап warmup

# Параметры
columns_left = ['Kp', 'Ki']
colors_left = ['green', 'blue']  # разные цвета для Kp и Ki
column_right = 'Error std norm'

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(2, 2, width_ratios=[1, 1.2], height_ratios=[1,1], wspace=0.3, hspace=0.4)

# --- Левая часть: Kp и Ki ---
for i, col in enumerate(columns_left):
    ax = fig.add_subplot(gs[i,0])
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])
    
    if neural_network_step <= step_df['Step'].max():
        ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
    
    ax.set_ylabel(col)
    ax.set_xlabel('Block')
    ax.grid(True, alpha=0.3)

# --- Правая часть: Error std norm ---
ax_right = fig.add_subplot(gs[:,1])  # занимает обе строки
ax_right.plot(step_df['Step'], step_df[column_right], alpha=0.8, linewidth=1.5, color='purple', label=column_right)

if neural_network_step <= step_df['Step'].max():
    ax_right.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

ax_right.set_xlabel('Block')
ax_right.set_ylabel(column_right)
ax_right.grid(True, alpha=0.3)
ax_right.legend()

fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.4)

## График для защиты

In [ ]:
print(neural_network_step)

In [ ]:
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

columns_left = ['Kp', 'Ki']
colors_left = ['green', 'blue']
column_right = 'Error std norm'
column_right_ru = 'СКВО напряжения\nна фотодетекторе'

time_min = (step_df['time'] - step_df['time'].min()) / 60.0

switch_time_min = None
if neural_network_step <= step_df['Step'].max():
    mask_exact = step_df['Step'] == neural_network_step
    if mask_exact.any():
        switch_time_min = float(time_min[mask_exact].iloc[0])
    else:
        switch_time_min = float(
            np.interp(neural_network_step, step_df['Step'].to_numpy(), time_min.to_numpy())
        )

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(
    2, 2,
    width_ratios=[1, 1.2],
    height_ratios=[1, 1],
    wspace=0.3, hspace=0.4
)

axes_left = []
for i, col in enumerate(columns_left):
    if i == 0:
        ax = fig.add_subplot(gs[i, 0])
    else:
        ax = fig.add_subplot(gs[i, 0], sharex=axes_left[0])
    axes_left.append(ax)

    ax.plot(time_min, step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])

    if switch_time_min is not None:
        ax.axvline(x=switch_time_min, color='red', linestyle='--', linewidth=2)

    ax.set_ylabel(col)
    ax.grid(False)

# Подпись X только на нижнем графике
axes_left[-1].set_xlabel('Время, мин')
for ax in axes_left[:-1]:
    ax.tick_params(labelbottom=False)

# Правый большой график
ax_right = fig.add_subplot(gs[:, 1])
ax_right.plot(
    time_min,
    step_df[column_right],
    alpha=0.8,
    linewidth=1.5,
    color='purple',
)

if switch_time_min is not None:
    ax_right.axvline(
        x=switch_time_min,
        color='red',
        linestyle='--',
        linewidth=2,
        label='Переключение на НС'
    )

ax_right.set_xlabel('Время, мин')
ax_right.set_ylabel(column_right_ru)
ax_right.grid(False)

fig.align_ylabels(axes_left)
fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.10, wspace=0.3, hspace=0.4)
plt.savefig("pid_tune_2coef.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec

def hsl_to_rgb_normalized(h, s, l):
    # colorsys использует HLS (не HSL)
    # поэтому порядок: (h, l, s)
    from colorsys import hls_to_rgb
    return hls_to_rgb(h / 360, l / 100, s / 100)

BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
GREEN_RGB = hsl_to_rgb_normalized(120, 72, 44)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)
PURPLE_RGB = hsl_to_rgb_normalized(279, 98, 76)
LIGHT_PURPLE_RGB = hsl_to_rgb_normalized(278, 100, 94)

GRAY_RGB = (123 / 255, 123 / 255, 123 / 255)

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

error_mean_normalization_factor = config.env.args.error_mean_normalization_factor
error_std_normalization_factor = config.env.args.error_std_normalization_factor

error_mean = step_df['Error mean norm'] * error_mean_normalization_factor
error_std = step_df['Error std norm'] * error_std_normalization_factor
rmse = np.sqrt(error_mean ** 2 + error_std ** 2) / 10 # старый масштаб

columns_left = ['Kp', 'Ki']
colors_left = [GREEN_RGB, BLUE_RGB]
column_right_ru = 'Среднеквадратическая\nошибка'

time_min = (step_df['time'] - step_df['time'].min()) / 60.0

neural_network_step = max(initial_collect_steps, exploration_steps) - 100 # этап warmup

switch_time_min = None
if neural_network_step <= step_df['Step'].max():
    mask_exact = step_df['Step'] == neural_network_step
    if mask_exact.any():
        switch_time_min = float(time_min[mask_exact].iloc[0])
    else:
        switch_time_min = float(
            np.interp(neural_network_step, step_df['Step'].to_numpy(), time_min.to_numpy())
        )

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(
    2, 2,
    width_ratios=[1, 1.2],
    height_ratios=[1, 1],
    wspace=0.3, hspace=0.4
)

# Левый столбец: 2 графика с общей осью X
axes_left = []
for i, col in enumerate(columns_left):
    if i == 0:
        ax = fig.add_subplot(gs[i, 0])
    else:
        ax = fig.add_subplot(gs[i, 0], sharex=axes_left[0])
    axes_left.append(ax)

    ax.plot(time_min, step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])

    if switch_time_min is not None:
        ax.axvline(x=switch_time_min, color='orange', linestyle='--', linewidth=5)

    ax.set_ylabel(col)
    ax.grid(False)

# Подпись X только на нижнем графике
axes_left[-1].set_xlabel('Время, мин')
for ax in axes_left[:-1]:
    ax.tick_params(labelbottom=False)

# Правый большой график
ax_right = fig.add_subplot(gs[:, 1])
ax_right.plot(
    time_min,
    rmse,
    alpha=0.8,
    linewidth=1.5,
    color=PURPLE_RGB,
)

if switch_time_min is not None:
    ax_right.axvline(
        x=switch_time_min,
        color='orange',
        linestyle='--',
        linewidth=5,
        label='Включение НС-контроллера'
    )

ax_right.set_xlabel('Время, мин')
ax_right.set_ylabel(column_right_ru)
ax_right.grid(False)

handles, labels = ax_right.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.95),
    ncol=2,
    frameon=False,
)

fig.align_ylabels(axes_left)
fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.10, wspace=0.3, hspace=0.4)

plt.savefig("pid_tune_2coef.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
nn_step_df = step_df[step_df['Step'] >= neural_network_step].copy()

if len(nn_step_df) > 0:
    plt.figure(figsize=(12, 5))
    plt.plot(nn_step_df['Step'], nn_step_df['Error std norm'], alpha=0.8, linewidth=0.8)
    
    for reset_step in reset_steps:
        if reset_step >= neural_network_step and reset_step <= nn_step_df['Step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5, label='Reset' if reset_step == reset_steps[0] else '')
    
    plt.title('Error std norm over Steps (NN phase)')
    plt.xlabel('Step')
    plt.ylabel('Error std norm')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=step_df['Kp'], y=step_df['Error std norm'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error Std Norm')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=step_df['Kp'], y=step_df['Error mean norm'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error mean norm')
cur_ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=step_df['Ki'], y=step_df['Error std norm'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error Std Norm')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=step_df['Ki'], y=step_df['Error mean norm'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error mean norm')
cur_ax.grid(True, alpha=0.3)


In [ ]:
corr_matrix = step_df[['Kp', 'Ki', 'Error mean norm', 'Error std norm', 'Reward']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


## Анализ процесса обучения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_train_log

# Новый формат: step time loss_q1 loss_q2 [actor_loss] buffer_size.
# Фильтруем строки-метрики и приводим к историческому набору колонок
# (actor_loss = NaN там, где актор не обновлялся).
loss_df = parse_train_log(TRAIN_LOG_PATH)
loss_df = (
    loss_df[loss_df[["loss_q1", "loss_q2", "buffer_size"]].notna().all(axis=1)]
    .reindex(columns=["step", "loss_q1", "loss_q2", "actor_loss", "buffer_size"])
    .reset_index(drop=True)
)
print(f"Загружено {len(loss_df)} записей из логов обучения")
print(f"Диапазон шагов обучения: {loss_df['step'].min()} - {loss_df['step'].max()}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(loss_df['step'], loss_df['loss_q1'], 'b-', alpha=0.7, label='Q1 Loss')
axes[0].set_title('Q1 Loss')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_df['step'], loss_df['loss_q2'], 'g-', alpha=0.7, label='Q2 Loss')
axes[1].set_title('Q2 Loss')
axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(loss_df['step'], loss_df['loss_q1'] + loss_df['loss_q2'], 'r--', alpha=0.7, label='Sum (Q1 + Q2)')
axes[2].set_title('Sum (Q1 + Q2)')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Loss')
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
actor_loss_df = loss_df[loss_df['actor_loss'].notna()]
if len(actor_loss_df) > 0:
    plt.plot(actor_loss_df['step'], actor_loss_df['actor_loss'], 'r-', alpha=0.7)
    plt.title('Actor Loss')
else:
    plt.text(0.5, 0.5, 'No actor loss data', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Actor Loss (no data)')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(loss_df['step'], loss_df['buffer_size'], 'm-', alpha=0.7)
plt.title('Buffer Size')
plt.xlabel('Step')
plt.ylabel('Size')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Анализ состояния установки


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_connection_log

# parse_connection_log отдаёт сырые записи SEND/READ (тип — в колонке 'event').
# Историческое поведение: i-я посылка (SEND) спаривается с i-м чтением (READ).
conn_raw = parse_connection_log(CONNECTION_LOG_PATH)

send_df = conn_raw[conn_raw["event"] == "SEND"][
    ["kp", "ki", "kd", "control_min", "control_max"]
].reset_index(drop=True)
send_df["step"] = range(len(send_df))

read_df = conn_raw[conn_raw["event"] == "READ"][
    ["process_variable", "control_output"]
].reset_index(drop=True)
read_df["step"] = range(len(read_df))

connection_df = send_df.merge(read_df, on="step", how="left")
print(f"Загружено {len(connection_df)} записей из логов соединения")
if len(connection_df) > 0:
    print(f"Диапазон шагов: {connection_df['step'].min()} - {connection_df['step'].max()}")

In [ ]:
neural_network_step = (neural_network_step - 100) * config.env.args.block_size
if len(connection_df) > 0:
    setpoint = config.env.args.setpoint
    
    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')
    plt.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint ({setpoint})')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('Process Variable')
    plt.xlabel('Step')
    plt.ylim(500, 1700)
    plt.ylabel('Process Variable')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('Control Output')
    plt.xlabel('Step')
    plt.ylabel('Control Output')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['kp'], 'r-', alpha=0.7, linewidth=0.8, label='Kp')
    plt.plot(connection_df['step'], connection_df['ki'], 'g-', alpha=0.7, linewidth=0.8, label='Ki')
    plt.plot(connection_df['step'], connection_df['kd'], 'b-', alpha=0.7, linewidth=0.8, label='Kd')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('PID Coefficients')
    plt.xlabel('Step')
    plt.ylabel('Coefficient Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Настройка шрифта
plt.rcParams.update({'font.size': 14})

# Определяем данные
columns_left = ['process_variable']
columns_right = ['control_output']
fig = plt.figure(figsize=(18, 6))
gs = GridSpec(1, 2, width_ratios=[1, 1], wspace=0.3)

ax_left = fig.add_subplot(gs[0,0])
ax_left.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=1.2, label='Process Variable')
ax_left.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint')
ax_left.set_xlim(left=100)

if neural_network_step <= connection_df['step'].max():
    ax_left.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

ax_left.set_xlabel('Step')
ax_left.set_ylabel('Process Variable')
ax_left.set_title('Process Variable')
ax_left.grid(True, alpha=0.3)
ax_left.legend()

ax_right = fig.add_subplot(gs[0,1])
ax_right.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=1.2, label='Control Output')
ax_right.set_xlim(left=100)

if neural_network_step <= connection_df['step'].max():
    ax_right.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)

ax_right.set_xlabel('Step')
ax_right.set_ylabel('Control Output')
ax_right.set_title('Control Output')
ax_right.grid(True, alpha=0.3)
ax_right.legend()

plt.tight_layout()
plt.show()


In [ ]:
if len(connection_df) > 0:
    corr_matrix = connection_df[['kp', 'ki', 'control_output', 'process_variable']].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix (Connection Data)')
    plt.tight_layout()
    plt.show()


In [ ]:
step_df = step_df.sort_values('time')
step_df['time_diff'] = step_df['time'].diff()
step_df['time_diff_ms'] = step_df['time_diff'] * 1000 
if len(step_df) > 0:
    step_df['time_relative'] = step_df['time'] - step_df['time'].iloc[0]
    step_df['time_relative_minutes'] = step_df['time_relative'] / 60

time_df, step_time_df = env_df.copy(), step_df.copy()

print("=== Статистика по времени ===")
print(f"Всего записей: {len(time_df)}")
print(f"Шагов: {len(time_df[time_df['Type'] == 'step'])}")
print(f"Reset событий: {len(time_df[time_df['Type'] == 'reset'])}")

if len(step_time_df) > 0:
    print(f"\n=== Статистика интервалов между шагами ===")
    print(step_time_df['time_diff_ms'].describe())
    print(f"\nМедианный интервал: {step_time_df['time_diff_ms'].median():.2f} мс")
    print(f"Средний интервал: {step_time_df['time_diff_ms'].mean():.2f} мс")
    print(f"Максимальный интервал: {step_time_df['time_diff_ms'].max():.2f} мс")
    print(f"Минимальный интервал: {step_time_df['time_diff_ms'].min():.2f} мс")
    
    plt.figure(figsize=(12, 5))
    plt.plot(step_time_df['Step'], step_time_df['time_diff_ms'], 'b-', alpha=0.7, linewidth=0.8)
    plt.title('Time Intervals Between Steps')
    plt.xlabel('Step')
    plt.ylabel('Time Interval (ms)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 5))
    plt.hist(step_time_df['time_diff_ms'].dropna(), bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Time Intervals Between Steps')
    plt.xlabel('Time Interval (ms)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
actor_loss_df = loss_df[loss_df['actor_loss'].notna()]
if len(actor_loss_df) > 0:
    plt.plot(actor_loss_df['step'], actor_loss_df['actor_loss'], 'r-', alpha=0.7)
    plt.title('Actor Loss (Kp and Ki)')
else:
    plt.text(0.5, 0.5, 'No actor loss data', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Actor Loss (no data)')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()